# T07 — Moscot: Optimal Transport for Single-Cell

**Tool:** `moscot` (Theis Lab)  
**Dataset:** Mouse pancreas (from T03), simulated time-course  
**Paper:** [Lange et al. 2023, Nature Methods](https://doi.org/10.1038/s41592-023-02066-3)

---

## Optimal Transport (OT) for single-cell — why?

RNA velocity (T03) infers directionality from the spliced/unspliced ratio — it doesn't require time labels. But it has limitations:
- Many genes don't have reliable velocity (low unspliced counts)
- It can be noisy for mature/quiescent cell states
- It can't use actual experimental time stamps when you have them

**Moscot** uses **optimal transport** to match cells across time points or conditions. The intuition: OT finds the minimum-cost "transport plan" T where T[i,j] = probability that cell i at time t₁ gave rise to cell j at time t₂. The "cost" is typically gene expression distance.

Moscot can also solve:
| Problem | What it does |
|---------|--------------|
| `TemporalProblem` | Cell fate mapping with experimental time labels |
| `LineageProblem` | Cell fate with lineage barcodes (LARRY, Waterfall) |
| `SpatialAlignmentProblem` | Align multiple spatial slides to a common coordinate system |
| `SpatialMappingProblem` | Map scRNA-seq cells to spatial coordinates (like TANGRAM) |
| `TranslationProblem` | Integrate unpaired multi-modal data |

## OT vs. velocity

| Approach | Signal | Best for |
|----------|--------|----------|
| RNA velocity (scVelo) | Spliced/unspliced ratio | Continuous differentiation without time labels |
| CellRank VelocityKernel | Same | Fate inference from velocity |
| **Moscot TemporalProblem** | Experimental time stamps | Labeled time-course experiments |
| **Moscot LineageProblem** | Barcode + time + velocity | Clonal tracing experiments |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import moscot
from moscot.problems import TemporalProblem
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'moscot {moscot.__version__}')

## 1. Load and Prepare Data

Moscot needs a `time_key` in `adata.obs` — a numeric or ordinal variable indicating the time point of each cell. Here we load the pancreas dataset from T03 and simulate time labels (real datasets like scNT-seq or time-course experiments have actual timestamps).

In [ ]:
# Load pancreas dataset (same as T03)
adata = scv.datasets.pancreas()
print(adata)
print('\nAnnotated clusters:', adata.obs['clusters'].unique().tolist())

In [ ]:
# Basic preprocessing
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

# Run velocity to get latent time (used as proxy for experimental time here)
scv.tl.recover_dynamics(adata, n_jobs=2)
scv.tl.velocity(adata, mode='dynamical')
scv.tl.velocity_graph(adata)
scv.tl.latent_time(adata)

In [ ]:
# Simulate discrete time labels from latent time
# (In a real experiment, this would be experimental day: 0, 3, 7, 14, etc.)
latent_time = adata.obs['latent_time']
bins = pd.qcut(latent_time, q=3, labels=['E1', 'E2', 'E3'])  # 3 pseudo time points
adata.obs['time_point'] = bins.astype(str)
adata.obs['time_numeric'] = bins.cat.codes.astype(float)  # moscot needs numeric

print('Time point distribution:')
print(adata.obs['time_point'].value_counts().sort_index())

sc.pl.umap(adata, color=['clusters', 'time_point', 'latent_time'],
           ncols=3, legend_loc='on data')

## 2. Moscot: TemporalProblem

The `TemporalProblem` models cell fate across time points using OT. It finds the transport plan T between consecutive time points that minimizes expression-space cost while respecting cell proliferation rates.

In [ ]:
# Setup the Temporal Problem
# time_key: obs column with numeric time values
# joint_attr: which representation to use for OT cost (X_pca recommended)
tp = TemporalProblem(adata)
tp = tp.prepare(
    time_key='time_numeric',
    joint_attr='X_pca',        # OT cost computed in PCA space
)
print(tp)

In [ ]:
# Solve: find optimal transport plan between each pair of consecutive time points
# epsilon: regularization strength (higher = smoother/more diffuse transport)
# tau_a, tau_b: marginal relaxation (handles unbalanced growth/death)
tp = tp.solve(
    epsilon=0.05,
    tau_a=0.95,
    tau_b=0.95,
    rank=-1,           # -1 = full rank (exact); positive int = low-rank approximation
)
print('Solved!')
print('Transport plans:', list(tp.solutions.keys()))

In [ ]:
# Inspect the transport matrix for one time transition
# T[i,j] = probability that cell i at t=0 gave rise to cell j at t=1
T_01 = tp[(0.0, 1.0)].transport_matrix
print(f'Transport matrix E1→E2: {T_01.shape}')
print('Row sums (should ≈ 1):', T_01.sum(axis=1)[:5])

## 3. Cell Fate Predictions

From the transport plan, Moscot computes **push-forward** (where do cells go?) and **pull-back** (where did cells come from?) distributions.

In [ ]:
# Push-forward: given early cells, predict where they end up
# Source: E1 cells; Target: E2 distribution
push = tp.push(
    source=0.0,
    target=1.0,
    data='clusters',    # obs column to push
    subset='Ductal',    # push only Ductal cells
    normalize=True,
)
# push is a distribution over E2 cells — which clusters do Ductal cells give rise to?
print('Ductal fate distribution at E2:')
print(push.value_counts().head(10))

In [ ]:
# Pull-back: which early cells gave rise to terminal Alpha cells?
pull = tp.pull(
    source=0.0,
    target=1.0,
    data='clusters',
    subset='Alpha',
    normalize=True,
)
print('Alpha cell ancestry (E1):')
print(pull.value_counts().head(10))

In [ ]:
# Compute fate probabilities for all terminal states
# (analogous to CellRank's compute_fate_probabilities)
cell_fates = tp.compute_fate_probabilities(
    terminal_states=adata.obs.loc[adata.obs['time_numeric'] == 2.0, 'clusters'],
)
# Each early cell gets a probability of reaching each terminal cluster
print('Fate probabilities shape:', cell_fates.shape)

## 4. Spatial Alignment (Bonus)

Moscot can also align multiple spatial transcriptomics slides to a common coordinate system — useful when you have serial sections or multiple patients.

In [ ]:
# Conceptual demo — spatial alignment
# In practice you would have two Visium slides:
#   adata_slide1 with .obsm['spatial'] coords
#   adata_slide2 with .obsm['spatial'] coords

from moscot.problems import SpatialAlignmentProblem

# sap = SpatialAlignmentProblem(adata1, adata2)
# sap = sap.prepare(batch_key='slide')       # obs column indicating which slide
# sap = sap.solve(epsilon=0.01, alpha=0.5)   # alpha: balance gene vs. spatial cost
# sap.align()                                # writes aligned coords to .obsm

print('SpatialAlignmentProblem is available for multi-slide integration.')
print('See: https://moscot.readthedocs.io/en/latest/notebooks/examples/spatial/200_alignment.html')

---
## Summary

```python
from moscot.problems import TemporalProblem

# 1. Prepare
tp = TemporalProblem(adata)
tp = tp.prepare(time_key='day', joint_attr='X_pca')

# 2. Solve
tp = tp.solve(epsilon=0.05, tau_a=0.95, tau_b=0.95)

# 3. Interrogate
tp.push(source=0, target=7, data='cell_type', subset='Stem')    # fate of Stem cells
tp.pull(source=0, target=7, data='cell_type', subset='Alpha')   # ancestry of Alpha
tp.compute_fate_probabilities(terminal_states=...)
```

| Moscot problem | Use case |
|----------------|----------|
| `TemporalProblem` | Time-course with labels (days, stages) |
| `LineageProblem` | Clonal tracing (LARRY barcodes) |
| `SpatialAlignmentProblem` | Align multiple spatial slides |
| `SpatialMappingProblem` | Map scRNA-seq to spatial (like TANGRAM) |
| `TranslationProblem` | Unpaired multi-modal integration |

**Moscot vs. CellRank VelocityKernel:** Moscot excels when you have experimental time labels. CellRank VelocityKernel excels when you have good velocity data but no time labels. CellRank 2 can also use Moscot's transport plans as a kernel (`moscot.RealTimeKernel`).

**Next:** T08 — scCODA for compositional analysis of cell type fractions.